# Home Credit Baseline - Binary Classification Model

This notebook builds a binary classification model using traditional, boosted and deep learning methodes to predict loan defaults based on home credit data and test its stability over time.

**Source:** https://www.kaggle.com/code/greysky/home-credit-baseline

## Step 1: Import Libraries and Set Paths

Import required libraries for data processing (polars, numpy), visualization (matplotlib, seaborn), and machine learning (scikit-learn, LightGBM).
Define paths to training, testing, and sample data directories.


In [ ]:
import gc
from glob import glob

import numpy as np
import pandas as pd
import polars as pl

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.base import BaseEstimator, ClassifierMixin

import lightgbm as lgb


TRAIN_DIR = "data/train"
TEST_DIR = "data/test"
SAMPLE_DIR = "data/sample"

## Step 2: Define Helper Classes

### Pipeline Class
Handles data preprocessing steps:
- **set_table_dtypes**: Cast columns to appropriate data types (Int32, Float64, Date, String)
- **handle_dates**: Convert date columns to days relative to decision date
- **filter_cols**: Remove columns with >95% missing values or low variance (categorical with 1 or >200 unique values)

In [ ]:
class Pipeline:
    @staticmethod
    def set_table_dtypes(df: pl.DataFrame):
        for col in df.columns:
            if col in ["case_id", "WEEK_NUM", "num_group1", "num_group2"]:
                df = df.with_columns(pl.col(col).cast(pl.Int32))
            elif col in ["date_decision"]:
                df = df.with_columns(pl.col(col).cast(pl.Date))
            elif col[-1] in ("P", "A"):
                df = df.with_columns(pl.col(col).cast(pl.Float64))
            elif col[-1] in ("M",):
                df = df.with_columns(pl.col(col).cast(pl.String))
            elif col[-1] in ("D",):
                df = df.with_columns(pl.col(col).cast(pl.Date))

        return df

    @staticmethod
    def handle_dates(df: pl.DataFrame): # Voor elke datumkolom, bereken het aantal dagen sinds de datum van beslissing
        for col in df.columns:
            if col[-1] in ("D",):
                df = df.with_columns(pl.col(col) - pl.col("date_decision"))
                df = df.with_columns(pl.col(col).dt.total_days())
                df = df.with_columns(pl.col(col).cast(pl.Float32))

        df = df.drop("date_decision", "MONTH")

        return df

    @staticmethod
    def filter_cols(df: pl.DataFrame):
        for col in df.columns:
            if col not in ["target", "case_id", "WEEK_NUM"]:
                isnull = df[col].is_null().mean()

                if isnull > 0.95:
                    df = df.drop(col)

        for col in df.columns:
            if (col not in ["target", "case_id", "WEEK_NUM"]) & (df[col].dtype == pl.String):
                freq = df[col].n_unique()

                if (freq == 1) | (freq > 200):
                    df = df.drop(col)

        return df
    
    @staticmethod
    def filter_correlated_cols(df: pd.DataFrame, threshold=0.95):
        """Remove one column from each pair of highly correlated columns (keeps non-numeric columns)"""
        # Protect important columns from being dropped
        protected_cols = ["target", "case_id", "WEEK_NUM"]
        
        # Get only numeric columns (excluding protected cols)
        numeric_cols = [col for col in df.select_dtypes(include=[np.number]).columns 
                       if col not in protected_cols]
        
        # Calculate correlation matrix (only upper triangle for efficiency)
        corr_matrix = df[numeric_cols].corr(method='pearson').abs()
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
        corr_upper = corr_matrix.where(mask)   

        to_drop = [col for col in corr_upper.columns if any(corr_upper[col] > threshold)]

        # kwil leeftijd houden
        if "age" in to_drop:
            to_drop.remove("age")
        
        print(f"Dropping {len(to_drop)} highly correlated columns (threshold={threshold})")
        return df.drop(columns=to_drop)

### Aggregator Class
Aggregates features by case_id at different depth levels. Creates maximum values for:
- Numeric features (columns ending in "P" or "A")
- Date features (columns ending in "D")
- String features (columns ending in "M")
- Other categorical features (columns ending in "T" or "L")
- Group count columns


In [ ]:
class Aggregator:
    @staticmethod
    def num_expr(df: pl.DataFrame):
        cols = [col for col in df.columns if col[-1] in ("P", "A")]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def date_expr(df: pl.DataFrame):
        cols = [col for col in df.columns if col[-1] in ("D",)]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def str_expr(df: pl.DataFrame):
        cols = [col for col in df.columns if col[-1] in ("M",)]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def other_expr(df: pl.DataFrame):
        cols = [col for col in df.columns if col[-1] in ("T", "L")]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def count_expr(df: pl.DataFrame):
        cols = [col for col in df.columns if "num_group" in col]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def get_exprs(df: pl.DataFrame):
        exprs = Aggregator.num_expr(df) + \
            Aggregator.date_expr(df) + \
            Aggregator.str_expr(df) + \
            Aggregator.other_expr(df) + \
            Aggregator.count_expr(df)

        return exprs

## Step 3: Data Loading Functions

- **read_file**: Reads a single parquet file and applies type casting and aggregation
- **read_files**: Reads multiple parquet files matching a glob pattern, concatenates them, and removes duplicates


In [ ]:
def read_file(path, depth=None):
    df = pl.read_parquet(path)
    df = df.pipe(Pipeline.set_table_dtypes)

    if depth in [1, 2]:
        df = df.group_by("case_id").agg(Aggregator.get_exprs(df))

    return df


def read_files(regex_path, depth=None):
    chunks = []
    for path in glob(str(regex_path)):
        df = pl.read_parquet(path)
        df = df.pipe(Pipeline.set_table_dtypes)

        if depth in [1, 2]:
            df = df.group_by("case_id").agg(Aggregator.get_exprs(df))

        chunks.append(df)

    df = pl.concat(chunks, how="vertical_relaxed")
    df = df.unique(subset=["case_id"])

    return df

## Step 4: Feature Engineering

Combines base features with features from different data depths:
- Extracts month and weekday from decision date
- Joins multiple feature tables on case_id
- Converts date columns to relative days and handles missing values


In [ ]:
def feature_eng(df_base: pl.DataFrame, depth_0, depth_1, depth_2) -> pl.DataFrame:
    df_base = (
        df_base
        .with_columns(
            month_decision=pl.col("date_decision").dt.month(),
            weekday_decision=pl.col("date_decision").dt.weekday(),
            year_decision=pl.col("date_decision").dt.year(),
        )
    )

    for i, df in enumerate(depth_0 + depth_1 + depth_2):
        df_base = df_base.join(df, how="left", on="case_id", suffix=f"_{i}")

    df_base = df_base.pipe(Pipeline.handle_dates)

    return df_base

## Step 5: Data Format Conversion

Converts polars DataFrame to pandas and converts object columns to categorical type for memory efficiency.


In [ ]:
def to_pandas(df_data: pl.DataFrame, cat_cols=None) -> pd.DataFrame:
    df_data: pd.DataFrame = df_data.to_pandas()

    if cat_cols is None:
        cat_cols = list(df_data.select_dtypes("object").columns)

    df_data[cat_cols] = df_data[cat_cols].astype("category")

    return df_data, cat_cols

## Step 6: Load and Prepare Training and Testing Data

Loads all training and testing data files, performs feature engineering, and prepares the dataset. Training data includes base information plus features from different depths (static, applications, tax registry, credit bureau, etc.).


In [ ]:
data_store = {
    "df_base": read_file(f"{SAMPLE_DIR}/train_base_sampled.parquet"),
    "depth_0": [
        read_file(f"{SAMPLE_DIR}/train_static_cb_0_sampled.parquet"),
        read_files(f"{SAMPLE_DIR}/train_static_0_*.parquet"),
    ],
    "depth_1": [
        read_files(f"{SAMPLE_DIR}/train_applprev_1_*.parquet", 1),
        read_file(f"{SAMPLE_DIR}/train_tax_registry_a_1_sampled.parquet", 1),
        read_file(f"{SAMPLE_DIR}/train_tax_registry_b_1_sampled.parquet", 1),
        read_file(f"{SAMPLE_DIR}/train_tax_registry_c_1_sampled.parquet", 1),
        read_files(f"{SAMPLE_DIR}/train_credit_bureau_a_1_*.parquet", 1),
        read_file(f"{SAMPLE_DIR}/train_credit_bureau_b_1_sampled.parquet", 1),
        read_file(f"{SAMPLE_DIR}/train_other_1_sampled.parquet", 1),
        read_file(f"{SAMPLE_DIR}/train_person_1_sampled.parquet", 1),
        read_file(f"{SAMPLE_DIR}/train_deposit_1_sampled.parquet", 1),
        read_file(f"{SAMPLE_DIR}/train_debitcard_1_sampled.parquet", 1),
    ],
    "depth_2": [
        read_file(f"{SAMPLE_DIR}/train_credit_bureau_b_2_sampled.parquet", 2),
        read_files(f"{SAMPLE_DIR}/train_credit_bureau_a_2_*.parquet", 2),
    ]
}

df_train = feature_eng(**data_store)
print("train data shape:\t", df_train.shape)

In [ ]:
data_store = {
    "df_base": read_file(f"{TEST_DIR}/test_base.parquet"),
    "depth_0": [
        read_file(f"{TEST_DIR}/test_static_cb_0.parquet"),
        read_files(f"{TEST_DIR}/test_static_0_*.parquet"),
    ],
    "depth_1": [
        read_files(f"{TEST_DIR}/test_applprev_1_*.parquet", 1),
        read_file(f"{TEST_DIR}/test_tax_registry_a_1.parquet", 1),
        read_file(f"{TEST_DIR}/test_tax_registry_b_1.parquet", 1),
        read_file(f"{TEST_DIR}/test_tax_registry_c_1.parquet", 1),
        read_files(f"{TEST_DIR}/test_credit_bureau_a_1_*.parquet", 1),
        read_file(f"{TEST_DIR}/test_credit_bureau_b_1.parquet", 1),
        read_file(f"{TEST_DIR}/test_other_1.parquet", 1),
        read_file(f"{TEST_DIR}/test_person_1.parquet", 1),
        read_file(f"{TEST_DIR}/test_deposit_1.parquet", 1),
        read_file(f"{TEST_DIR}/test_debitcard_1.parquet", 1),
    ],
    "depth_2": [
        read_file(f"{TEST_DIR}/test_credit_bureau_b_2.parquet", 2),
        read_files(f"{TEST_DIR}/test_credit_bureau_a_2_*.parquet", 2),
    ]
}

df_test = feature_eng(**data_store)
print("test data shape:\t", df_test.shape)

## Step 7: Filter Features

Removes low-information columns from training data and keeps only the same features in test data.

Set an age rang of 18-100 bcs there were outliers/ages that are not possible

In [ ]:
df_train = df_train.pipe(Pipeline.filter_cols)
df_test = df_test.select([col for col in df_train.columns if col != "target"])

print("train data shape:\t", df_train.shape)
print("test data shape:\t", df_test.shape)

In [ ]:
df_train = df_train.with_columns(
    age=(abs(pl.col("dateofbirth_337D"))/365).round().cast(pl.UInt8))
df_test = df_test.with_columns(
    age=(abs(pl.col("dateofbirth_337D"))/365).round().cast(pl.UInt8))

df_train = df_train.filter((pl.col("age") >= 18) & (pl.col("age") <= 100))
df_test = df_test.filter((pl.col("age") >= 18) & (pl.col("age") <= 100))

print("train data shape:\t", df_train.shape)
print("test data shape:\t", df_test.shape)

## Step 8: Convert to Pandas Format

Converts both datasets to pandas DataFrames and converts categorical columns to category dtype for memory efficiency.


In [ ]:
df_train, cat_cols = to_pandas(df_train)
df_test, cat_cols = to_pandas(df_test, cat_cols)

# # Remove highly correlated columns (keeps all non-numeric columns) - TODO: misschien toch niet doen, want sommige modellen kunnen hier wel mee omgaan en het kan ook nuttige info bevatten (bv. max_payment_0_P en max_payment_1_P kunnen sterk gecorreleerd zijn maar toch allebei nuttig)
# df_train = Pipeline.filter_correlated_cols(df_train, threshold=0.95)
# df_test = df_test[[col for col in df_train.columns if col != "target"]]

# print("train data shape after correlation filter:\t", df_train.shape)
# print("test data shape after correlation filter:\t", df_test.shape)

del data_store
gc.collect()

## Step 9: Exploratory Data Analysis - Target Distribution Over Time

Visualizes how the target variable (loan default rate) changes across weeks to understand temporal patterns.


In [ ]:
sns.lineplot(
    data=df_train,
    x="WEEK_NUM",
    y="target",
)
plt.show()

In [ ]:
plt.scatter(data=df_train, x="age", y="target")
plt.xlabel("age")
plt.ylabel("target")

## Step 10: Stability Comparison Across 3 Simple Models

Train and compare weekly stability for:
1. Logistic Regression (traditional)
2. XGBoost (boosted)
3. Neural Network (deep learning)

Categorical features are handled using one-hot encoding in sklearn pipelines.

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LinearRegression

def compute_weekly_gini_stability(model, X, y, weeks):
    eval_df = pd.DataFrame({
        "WEEK_NUM": weeks.values,
        "target": y.values,
    })
    eval_df["pred"] = model.predict_proba(X)[:, 1]

    weekly_rows = []
    for week, grp in eval_df.groupby("WEEK_NUM", sort=True):
        # AUC/Gini are undefined if only one class is present in that week.
        if grp["target"].nunique() < 2:
            continue

        auc = roc_auc_score(grp["target"], grp["pred"])
        gini = 2.0 * auc - 1.0
        weekly_rows.append({"WEEK_NUM": week, "auc": auc, "gini": gini})

    weekly_gini = pd.DataFrame(weekly_rows).sort_values("WEEK_NUM").reset_index(drop=True)

    if len(weekly_gini) < 2:
        raise ValueError("Not enough valid weeks to fit a linear trend.")

    x = weekly_gini[["WEEK_NUM"]].values
    y_gini = weekly_gini["gini"].values

    reg = LinearRegression().fit(x, y_gini)
    a = float(reg.coef_[0])
    b = float(reg.intercept_)
    trend = reg.predict(x)

    residuals = y_gini - trend
    falling_rate = min(0.0, a)
    stability_metric = float(y_gini.mean() + 88.0 * falling_rate - 0.5 * residuals.std(ddof=0))

    summary = {
        "mean_gini": float(y_gini.mean()),
        "slope_a": a,
        "intercept_b": b,
        "falling_rate": falling_rate,
        "residual_std": float(residuals.std(ddof=0)),
        "stability_metric": stability_metric,
        "num_valid_weeks": int(len(weekly_gini)),
    }
    return weekly_gini, summary, trend

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

X = df_train.drop(columns=["target", "case_id", "WEEK_NUM"])
y = df_train["target"]
weeks = df_train["WEEK_NUM"]

# Split columns by dtype
cat_features = X.select_dtypes(include=["category", "object"]).columns.tolist()
num_features = X.columns.difference(cat_features).tolist()

# Sparse preprocessing for linear / tree models
sparse_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(with_mean=False), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_features),
    ],
    remainder="drop",
)

# Dense preprocessing for MLP
dense_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
    ],
    remainder="drop",
)

models = {
    "LogisticRegression": SkPipeline([
        ("prep", sparse_preprocessor),
        ("model", LogisticRegression(
            C=1.0,
            solver="saga",
            max_iter=300,
            random_state=42,
        )),
    ]),
    "XGBoost": SkPipeline([
        ("prep", sparse_preprocessor),
        ("model", XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            random_state=42,
            n_jobs=-1,
            enable_categorical=False,
        )),
    ]),
    "NeuralNetwork": SkPipeline([
        ("prep", dense_preprocessor),
        ("model", MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            alpha=1e-4,
            learning_rate_init=1e-3,
            max_iter=80,
            random_state=42,
            early_stopping=True,
            validation_fraction=0.1,
        )),
    ]),
}

model_summaries = []
weekly_gini_by_model = {}
trend_by_model = {}

for model_name, model_obj in models.items():
    print(f"Training {model_name}...")
    model_obj.fit(X, y)
    wk_gini, summary, trend_vals = compute_weekly_gini_stability(model_obj, X, y, weeks)

    summary_row = {"model": model_name, **summary}
    model_summaries.append(summary_row)
    weekly_gini_by_model[model_name] = wk_gini
    trend_by_model[model_name] = trend_vals

comparison_3models_df = pd.DataFrame(model_summaries)[[
    "model",
    "stability_metric",
    "mean_gini",
    "slope_a",
    "falling_rate",
    "residual_std",
    "num_valid_weeks",
]]

comparison_3models_df = comparison_3models_df.sort_values(
    "stability_metric", ascending=False
).reset_index(drop=True)

display(comparison_3models_df)

plt.figure(figsize=(12, 5))
for model_name in comparison_3models_df["model"]:
    wk = weekly_gini_by_model[model_name]
    tr = trend_by_model[model_name]
    plt.plot(wk["WEEK_NUM"], wk["gini"], marker="o", linewidth=1.5, label=f"{model_name} Gini")
    plt.plot(wk["WEEK_NUM"], tr, linestyle="--", linewidth=1.5, label=f"{model_name} trend")

plt.xlabel("WEEK_NUM")
plt.ylabel("Gini")
plt.title("Weekly Gini Stability: Logistic Regression vs XGBoost vs Neural Network")
plt.legend(ncol=2)
plt.grid(alpha=0.3)
plt.show()